## Running from Collab:


In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [3]:
import subprocess
import time
# Start the server as a background process
process = subprocess.Popen("ollama serve", shell=True)

# Wait a few seconds for the server to initialize
time.sleep(5)

In [ ]:
!ollama pull llama3.2

In [ ]:
!pip install chromadb


In [ ]:
#import os
import chromadb
#import requests
from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
#from PyPDF2 import PdfReader
#from google.colab import userdata
#from google import genai
from sentence_transformers import CrossEncoder

# Knowledge Base:

### Loading previous knowledge base:
-if loading previous knowledge base, just run the following cell   
-if building knowledge base, run all cells in this section

In [ ]:
# chroma_client = chromadb.PersistentClient(path='/content/')
# collection = client.create_collection(
#     name="my_collection",
#     embedding_function=OpenAIEmbeddingFunction(
#         model_name="text-embedding-3-small"
#         api_key_env_var=OPENAI_API_KEY
#     )
# )

chroma_client = chromadb.PersistentClient(path='./chroma_db')
collection = chroma_client.get_or_create_collection(name="test_collection")

In [ ]:
def get_text_txt_md(file_path: str) -> str:
  with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()
  return text

def get_text_pdf(pdf_path: str) -> str:
    """Extract raw text from a PDF file."""
    try:
        reader = PdfReader(pdf_path)
        return " ".join(page.extract_text() for page in reader.pages if page.extract_text())
    except Exception as e:
        raise RuntimeError(f"Error reading PDF: {e}")

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 25) -> list[str]:
    """Split text into manageable chunks for embeddings."""
    words = text.split()
    return [" ".join(words[i-overlap:i+chunk_size]) for i in range(overlap, len(words), chunk_size - overlap)]

In [ ]:
files_paths = ['toy_rag_data/Snake_wikiWikipedia.pdf', 'toy_rag_data/cool_math.pdf', 'toy_rag_data/its_nice_that.txt']
chunks = []
text = ""
for file in files_paths:
  if "pdf" in file:
    text = get_text_pdf(file)
  else:
    text = get_text_txt_md(file)
  chunks += chunk_text(text)

collection.upsert(
    documents=chunks,
    ids=[f"id{i}" for i in range(len(chunks))]
)

In [ ]:
#test retrieve:
collection.query(
      query_texts=["Who was the original creator?"],
      n_results=4
  )

# Generation:

In [ ]:
rr_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
def rerank(chunks, prompt, k, rr_model){
    return rr_model.rank(query, chunks, return_documents=True, top_k=3)
}

In [ ]:
def call_llm(name: str, model: str, prompt: str, client):
  if name == "gemini":
    try:
      response = client.interactions.create(
          model=model,
          input=prompt
      )
      return response.output_text
    except Exception as e:
        raise RuntimeError(f"Gemini LLM query failed: {e}")
  if name == "ollama":
    try:
      response = client.chat.completions.create(
          model=model,
          messages=[
              {"role": "user", "content": prompt}
          ]
      )
      return response.choices[0].message.content
    except Exception as e:
        raise RuntimeError(f"Ollama LLM query failed: {e}")

In [ ]:
def gen_response(query,  client, rr_model, collection, k, kp):
  k_chunks = collection.query(
      query_texts=[query],
      n_results=k
  )
  kp_chunks = rr_model.rank(query, chunks, return_documents=True, top_k=kp);
  kp_chunks = [item["text"] for item in kp_chunks]
  context = "\n".join(kp_chunks)
  prompt = f"{query} Only use the following context to answer this question. Clearly state when the context does not contain the answer: {context}"
  return call_llm("ollama", "gpt-oss:20b", prompt, client), k_chunks, kp_chunks


In [ ]:
client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)
rr_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
query = "When was the snake game made?"
response, k_chunks, kp_chunks = gen_response(query, client, rr_model, collection, 10, 3)
print(response)
print(kp_chunks)

In [ ]:
from openai import OpenAI
import ollama
ollama.pull('llama3.2')

client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)

responses_result = client.responses.create(
  model='llama3.2',
  input='repeat this sentence: this is a test',
)
print(responses_result.output_text)

# Testing Section

## TEST Retreival
* Tests retreival and reranking
* Does not test chunking
* Using the BeIR HotpotQA benchmark (corpus, questions and answers), we will not chunk because the corpus texts are already short.

In [ ]:
from datasets import load_dataset
from sentence_transformers import CrossEncoder
import pandas as pd
import chromadb

ds = load_dataset("BeIR/hotpotqa-generated-queries")
corpus = ds["train"][:1000]
chroma_client = chromadb.PersistentClient(path='chroma_db/test_db')
collection = chroma_client.get_or_create_collection(name="test_BeIR_collection")

In [ ]:
file_name = "ettin-reranker-32m-v1"
rr_name = "cross-encoder/ettin-reranker-32m-v1"
rr_model = CrossEncoder(rr_name)

def call_reranker(rr_name, rr_model, query, docs):
    if rr_name == "cross-encoder/ms-marco-MiniLM-L-6-v2" or rr_name == "cross-encoder/ettin-reranker-32m-v1":
        scores = rr_model.rank(documents=docs, query=query)
        return scores
    
        # pairs = [(query, doc) for doc in docs]
        # scores = rr_model.rank(pairs)
        # return scores
    return None

Loading weights: 100%|██████████| 62/62 [00:00<00:00, 9882.08it/s]


In [20]:
# only run this cell if you haven't loaded the BeIR corpus into the collection yet
collection.upsert(
    documents = corpus['text'],
    ids = corpus['_id'],
    metadatas=[{"title": title} for title in corpus["title"]]
)

In [ ]:
collection.query(
      query_texts=["who is the son of Telamon"],
      n_results=10
  )

In [25]:
res_df = pd.DataFrame(columns=[f"result_{i}" for i in range(1, 11)])
rr_df = pd.DataFrame(columns=[f"result_{i}" for i in range(1, 11)])
res_df.index.name = "query_id"

for i, q in enumerate(corpus['query'][:100]):
  res = collection.query(
        query_texts=[q],
        n_results=10
    )
  #pairs = [(q, doc) for doc in res['documents'][0]]
  rr_res = call_reranker(rr_name, rr_model, q, res['documents'][0])
  rr_df.loc[corpus['_id'][i]] = [item["score"] for item in rr_res]
  res_df.loc[corpus['_id'][i]] = res['ids'][0]

In [ ]:
def get_rank(row, correct_id):
  for i in range(0, 10):
      if row.iloc[i] == correct_id:
          return i + 1
  return None

def get_highest_rank(row):
  highest = row.iloc[0]
  highest_i = 0
  for i in range(1, 10):
      if row.iloc[i] > highest:
        highest = row.iloc[i]
        highest_i = i
  return highest_i + 1

res_df["rank"] = [
    get_rank(
        res_df.loc[query_id],
        query_id
    )
    for query_id in res_df.index
]

rr_df["highest_rank"] = [
    get_highest_rank(
        rr_df.loc[query_id]
    )
    for query_id in rr_df.index
]

rr_df["correct"] = [
    (rr_df.loc[query_id]['highest_rank'] == res_df.loc[query_id]['rank'])
    for query_id in rr_df.index
]



In [ ]:
rr_df["correct_score"] = [
    rr_df.loc[query_id].iloc[res_df.loc[query_id]['rank'] - 1]
    for query_id in rr_df.index
]



In [27]:
res_df

,result_1,result_2,result_3,result_4,result_5,result_6,result_7,result_8,result_9,result_10,rank
query_id,,,,,,,,,,,
12,12,1023,1738,1360,1748,1732,1737,1423,1734,1361,1
25,25,1653,876,2383,1338,2088,1485,892,1291,1028,1
39,39,1650,904,1383,573,1578,1210,1921,1394,1645,1
290,290,929,2166,960,1171,2170,670,2333,1400,2553,1
303,303,1285,1484,1286,1661,1644,803,2382,1649,1854,1
...,...,...,...,...,...,...,...,...,...,...,...
764,764,1840,1547,1543,1144,1549,1523,779,1556,1347,1
765,765,2457,2023,1303,1902,677,1747,1664,2075,1579,1
766,766,2349,1160,1807,1170,2125,1169,2014,713,1335,1


In [41]:
rr_df

,result_1,result_2,result_3,result_4,result_5,result_6,result_7,result_8,result_9,result_10,highest_rank,correct,correct_score
12,11.862968,9.092825,8.030506,7.977765,7.920308,7.720950,7.498271,7.386648,7.350549,7.255938,1,True,11.862968
25,10.339661,7.208236,7.099841,7.036287,6.644602,6.303551,5.551398,3.728289,-0.328843,-1.136695,1,True,10.339661
39,11.981813,6.070806,6.001335,5.909017,5.858575,5.364528,5.264916,5.201673,5.154221,4.909460,1,True,11.981813
290,10.473831,10.196280,9.205974,8.424699,8.248303,8.028863,7.961621,7.815057,7.765273,7.437471,1,True,10.473831
303,11.279359,9.558034,9.024461,8.125568,7.849253,7.733062,7.713409,7.623165,7.268131,6.842691,1,True,11.279359
...,...,...,...,...,...,...,...,...,...,...,...,...,...
764,10.730493,4.580027,4.507796,4.467965,4.409800,4.063651,3.963649,3.672529,3.570230,3.543754,1,True,10.730493
765,10.787915,6.499273,6.050671,5.736526,5.549923,4.139058,4.136932,3.862422,3.199518,2.824301,1,True,10.787915
766,11.215271,9.622183,9.034743,8.985269,8.904328,8.766132,8.747587,8.597581,8.587502,8.073304,1,True,11.215271
771,11.387043,10.856821,8.235101,8.040145,7.861691,7.775639,7.419187,7.294374,7.222813,7.121802,1,False,10.856821


In [29]:
res_df["rank"].mean()

np.float64(1.18)

In [34]:
rr_df["correct"].mean()

np.float64(0.87)

In [32]:
rr_df['highest_rank'].unique()

array([1])

In [33]:
rr_df.to_csv(f'test_ds/{file_name}.csv', index=False)